# `shire.ipynb` — Reconciliation & QA walkthrough

This notebook demonstrates the two main analytical features of the `resume_rdf` library
using the fictional consultant CVs in `shire/` as test data:

| File | Person | Notes |
|------|--------|-------|
| `shire/frodo_baggins_cv.md` | Frodo Baggins | Senior management consultant, 6 projects |
| `shire/sam_gamgee_cv1.md` | Sam Gamgee | Data engineering consultant, chronological framing |
| `shire/sam_gamgee_cv2.md` | Sam Gamgee | Same career, ESG/sustainability framing |

Frodo and Sam share **3 projects** (Northern Energy Holdings, Mordor Industrial Group,
White Council Capital), described from different roles and perspectives.
Sam's two CV versions describe the same projects with different emphasis and wording —
making both a cross-person and a within-person deduplication test.

**Pre-requisite:** `pip install -e ".[all]"` and set `ANTHROPIC_API_KEY`.

In [24]:
import os
import shutil
from pathlib import Path

# Set API key if not already in environment

SHIRE_DIR  = Path("shire") 
TTL_DIR    = SHIRE_DIR / "shire_ttl"
TTL_DIR.mkdir(exist_ok=True)

cvs = {
    "frodo":  SHIRE_DIR / "frodo_baggins_cv.md",
    "sam_v1": SHIRE_DIR / "sam_gamgee_cv1.md",
    "sam_v2": SHIRE_DIR / "sam_gamgee_cv2.md",
}
print("Input files:")
for name, p in cvs.items():
    print(f"  {name}: {p}  (exists: {p.exists()})")

Input files:
  frodo: shire/frodo_baggins_cv.md  (exists: True)
  sam_v1: shire/sam_gamgee_cv1.md  (exists: True)
  sam_v2: shire/sam_gamgee_cv2.md  (exists: True)


---
## Section 1 — Entity Reconciliation

Goal: convert the three markdown CVs to Turtle RDF, then find and merge
entities (projects, employers) that refer to the same real-world thing
but carry slightly different names or IRIs across files.

### 1.1 — Convert markdown CVs to Turtle RDF

`generate_graph_from_file` calls the Claude API and returns a Turtle string.
Responses are cached in `cache/` (SHA-256 keyed), so re-running is free.

In [25]:
from resume_rdf import generate_graph_from_file, count_triples

ttl_paths = {}
for name, cv_path in cvs.items():
    out_path = TTL_DIR / f"{name}.ttl"
    if out_path.exists():
        print(f"{name}: using cached {out_path}  ({count_triples(out_path.read_text())} triples)")
    else:
        print(f"{name}: generating from {cv_path} ...", end="", flush=True)
        turtle, usage = generate_graph_from_file(
            cv_path,
            extra_context="UK consultant CVs, energy and data engineering sectors. Output in English.",
            api_key = os.getenv("ANTHROPIC_API_KEY")
        )
        out_path.write_text(turtle)
        print(f"  {count_triples(turtle)} triples, {usage['output_tokens']:,} tokens")
    ttl_paths[name] = out_path

print("\nAll TTL files ready:")
for name, p in ttl_paths.items():
    print(f"  {name}: {p}")

frodo: using cached shire/shire_ttl/frodo.ttl  (264 triples)
sam_v1: using cached shire/shire_ttl/sam_v1.ttl  (311 triples)
sam_v2: using cached shire/shire_ttl/sam_v2.ttl  (362 triples)

All TTL files ready:
  frodo: shire/shire_ttl/frodo.ttl
  sam_v1: shire/shire_ttl/sam_v1.ttl
  sam_v2: shire/shire_ttl/sam_v2.ttl


### 1.2 — Extract labelled entities

`load_entities` parses each TTL file with rdflib and returns every
`cvx:Project` (labelled by `cvx:projectName`) and `cv:Company`
(labelled by `cv:Name`) node across all files.

In [26]:
from resume_rdf import load_entities

entities = load_entities(list(ttl_paths.values()))
print(f"Extracted {len(entities)} entities total\n")

for kind in ("project", "company"):
    subset = [e for e in entities if e.kind == kind]
    print(f"{kind.upper()} ({len(subset)}):")
    for e in subset:
        print(f"  [{e.source.name}]  {e.label!r}")
    print()

Extracted 46 entities total

PROJECT (20):
  [frodo.ttl]  'White Council Capital — ESG Reporting & Analytics Platform'
  [frodo.ttl]  'Mordor Industrial Group — Supply Chain Resilience Programme'
  [frodo.ttl]  'Northern Energy Holdings — Smart Grid Modernisation Programme'
  [frodo.ttl]  'Shire Financial Services — Regulatory Compliance Transformation'
  [frodo.ttl]  'Grey Havens Logistics — Digital Operations Review'
  [sam_v1.ttl]  'White Council Capital — ESG Reporting & Analytics Platform'
  [sam_v1.ttl]  'Mordor Industrial Group — Supply Chain Resilience Programme'
  [sam_v1.ttl]  'Northern Energy Holdings — Smart Grid Modernisation Programme'
  [sam_v1.ttl]  'Rivendell Health Systems — Clinical Data Pipeline Modernisation'
  [sam_v1.ttl]  'Rohan Agricultural Cooperative — Precision Farming Analytics Platform'
  [sam_v1.ttl]  'Gondor Municipal Authority — Infrastructure Monitoring & Reporting Platform'
  [sam_v1.ttl]  'rdflib Contributor — Turtle Serialiser Fixes'
  [sam_v1.ttl] 

### 1.3 — Find near-duplicate pairs

`find_matches` computes pairwise `difflib.SequenceMatcher` similarity
for every cross-file pair of the same entity type and returns those
above `threshold`.  We use 0.65 here to surface the shared-project
candidates even if the Claude parser worded them slightly differently.

In [27]:
from resume_rdf import find_matches

matches = find_matches(entities, threshold=0.65)
print(f"Found {len(matches)} candidate pair(s)  (threshold=65%):\n")

for i, m in enumerate(matches, 1):
    print(f"[{i}]  {m.score.as_integer_ratio()}  {m.score:.0%}")
    print(f"     A: {m.a.label!r:50}  ({m.a.source.name})")
    print(f"     B: {m.b.label!r:50}  ({m.b.source.name})")
    print(f"     A IRI: {m.a.iri}")
    print(f"     B IRI: {m.b.iri}")
    print()

Found 12 candidate pair(s)  (threshold=65%):

[1]  (1, 1)  100%
     A: 'White Council Capital — ESG Reporting & Analytics Platform'  (frodo.ttl)
     B: 'White Council Capital — ESG Reporting & Analytics Platform'  (sam_v1.ttl)
     A IRI: http://example.org/cv/proj_wcc_esg_2024
     B IRI: http://example.org/cv/proj_esg_white_council

[2]  (1, 1)  100%
     A: 'Mordor Industrial Group — Supply Chain Resilience Programme'  (frodo.ttl)
     B: 'Mordor Industrial Group — Supply Chain Resilience Programme'  (sam_v1.ttl)
     A IRI: http://example.org/cv/proj_mordor_supply_chain_2022
     B IRI: http://example.org/cv/proj_mordor_supply_chain

[3]  (1, 1)  100%
     A: 'Northern Energy Holdings — Smart Grid Modernisation Programme'  (frodo.ttl)
     B: 'Northern Energy Holdings — Smart Grid Modernisation Programme'  (sam_v1.ttl)
     A IRI: http://example.org/cv/proj_northern_energy_grid_2021
     B IRI: http://example.org/cv/proj_northern_energy_smartgrid

[4]  (1, 1)  100%
     A: 'White

### 1.4 — Interactive reconciliation

`reconcile_interactive` presents each candidate pair at the terminal
and asks `[y/n/q(uit)]`.  It can't run interactively inside a notebook,
so the cell below shows what a typical session looks like.

To run it for real, use the CLI:
```bash
cv-reconcile shire_ttl/frodo.ttl shire_ttl/sam_v1.ttl shire_ttl/sam_v2.ttl --threshold 0.65
```

**Expected terminal session:**
```
Loading 3 file(s)…
  Extracted 28 entities (18 projects, 10 companies).

Found 6 candidate pair(s) at ≥65% similarity:

── [1/6]  PROJECT  (similarity 94%) ──
  A: 'Smart Grid Modernisation'            :proj_smartgrid_2022
     (frodo.ttl)
  B: 'Smart Grid Modernisation Programme'  :proj_smart_grid_modernisation
     (sam_v1.ttl)
  Canonical if merged → A  (:proj_smartgrid_2022)
  Same entity? [y/n/q(uit)] y
  ✓ Will rewrite  :proj_smart_grid_modernisation  →  :proj_smartgrid_2022

── [2/6]  PROJECT  (similarity 91%) ──
  A: 'Smart Grid Modernisation'            :proj_smartgrid_2022
     (frodo.ttl)
  B: 'Smart Grid Modernisation — Northern Energy Holdings'  :proj_smart_grid_neh
     (sam_v2.ttl)
  Canonical if merged → A  (:proj_smartgrid_2022)
  Same entity? [y/n/q(uit)] y
  ✓ Will rewrite  :proj_smart_grid_neh  →  :proj_smartgrid_2022

── [3/6]  PROJECT  (similarity 88%) ──
  A: 'Supply Chain Resilience Programme'   :proj_mordor_supply_chain
     (frodo.ttl)
  B: 'Supply Chain Resilience'             :proj_supply_chain
     (sam_v1.ttl)
  Same entity? [y/n/q(uit)] y

... (3 more pairs)

Applying 5 merge(s) across 3 file(s)…
  frodo.ttl:  0 triple(s) rewritten
  sam_v1.ttl: 12 triple(s) rewritten
  sam_v2.ttl: 17 triple(s) rewritten
Done.
```

### 1.5 — Apply reconciliation programmatically

For automated or batch use, call `apply_mapping` directly with
the `mapping` dict you build from confirmed matches.

Below we auto-accept all matches above threshold and apply them to
working copies of the TTL files, then verify the shared projects
now share a single IRI in the merged graph.

In [28]:
from resume_rdf import apply_mapping, load_entities, find_matches

# Work on copies so the originals stay clean for re-use
rec_dir = SHIRE_DIR / Path("shire_reconciled")
if rec_dir.exists():
    shutil.rmtree(rec_dir)
shutil.copytree(TTL_DIR, rec_dir)
rec_paths = sorted(rec_dir.glob("*.ttl"))

# Build mapping from all matches (auto-yes)
rec_entities = load_entities(rec_paths)
rec_matches  = find_matches(rec_entities, threshold=0.65)

mapping = {}
for m in rec_matches:
    canon_a = mapping.get(m.a.iri, m.a.iri)
    canon_b = mapping.get(m.b.iri, m.b.iri)
    if canon_a != canon_b:
        mapping[canon_b] = canon_a

print(f"Applying {len(mapping)} IRI substitution(s):")
for old, new in mapping.items():
    print(f"  {str(old).rsplit('/', 1)[-1]}  →  {str(new).rsplit('/', 1)[-1]}")

results = apply_mapping(rec_paths, mapping)
print()
for path, count in sorted(results.items()):
    print(f"  {path.name}: {count} triple(s) rewritten")

Applying 9 IRI substitution(s):
  proj_esg_white_council  →  proj_wcc_esg_2024
  proj_mordor_supply_chain  →  proj_mordor_supply_chain_2022
  proj_northern_energy_smartgrid  →  proj_northern_energy_grid_2021
  company_white_council_capital  →  company_white_council
  company_univ_leeds  →  company_uni_leeds
  proj_esg_platform_wcc_2024  →  proj_wcc_esg_2024
  proj_fhir_rivendell_2025  →  proj_rivendell_fhir
  proj_precision_farming_rohan_2017  →  proj_rohan_farming
  company_rivendell_health  →  company_rivendell_partners

  sam_v1.ttl: 58 triple(s) rewritten
  sam_v2.ttl: 69 triple(s) rewritten


In [29]:
from rdflib import Graph, Namespace

_CVX = Namespace("http://example.org/cv-extension#")

# Load all reconciled files into one merged graph
merged = Graph()
for p in rec_paths:
    merged.parse(str(p), format="turtle")

print(f"Merged graph: {len(merged)} triples\n")
print("All cvx:Project IRIs in the merged graph:")

project_iris = set()
for proj_iri in merged.subjects(None, None):
    names = list(merged.objects(proj_iri, _CVX.projectName))
    if names:
        project_iris.add((str(proj_iri), str(names[0])))

for iri, name in sorted(project_iris, key=lambda x: x[1]):
    short = iri.rsplit("/", 1)[-1]
    print(f"  :{short:45}  {name!r}")

print("\nShared projects (IRI appears in > 1 source file):")
# Count how many source files contain each project IRI
from collections import Counter
iri_sources = Counter()
for p in rec_paths:
    g = Graph()
    g.parse(str(p), format="turtle")
    for iri, _, _ in g.triples((None, _CVX.projectName, None)):
        iri_sources[str(iri)] += 1

for iri, count in sorted(iri_sources.items(), key=lambda x: -x[1]):
    if count > 1:
        name = next(merged.objects(iri, _CVX.projectName), "?")
        short = iri.rsplit("/", 1)[-1]
        print(f"  :{short:45}  {name!r}  (appears in {count} files)")

Merged graph: 824 triples

All cvx:Project IRIs in the merged graph:
  :proj_gondor_infra                              'Gondor Municipal Authority — Infrastructure Monitoring & Reporting Platform'
  :proj_grey_havens_ops_2019                      'Grey Havens Logistics — Digital Operations Review'
  :proj_mordor_supply_chain_2022                  'Mordor Industrial Group — Supply Chain Resilience Programme'
  :proj_northern_energy_grid_2021                 'Northern Energy Holdings — Smart Grid Modernisation Programme'
  :proj_rivendell_fhir                            'Rivendell Health Systems — Clinical Data Pipeline Modernisation'
  :proj_rohan_farming                             'Rohan Agricultural Cooperative — Precision Farming Analytics Platform'
  :proj_shire_financial_mifid_2018                'Shire Financial Services — Regulatory Compliance Transformation'
  :proj_smart_grid_neh_2021                       'Smart Grid Modernisation'
  :proj_supply_chain_mordor_2022            

---
## Section 2 — CV Audit & QA

Goal: inspect each Turtle CV for missing or incomplete fields and
demonstrate how to fill them in using `update_field`.

`audit_experience` checks every `cv:WorkHistory` and linked `cvx:Project`
for required predicates and returns a `Question` for each gap.

### 2.1 — Audit all three CVs

In [30]:
from resume_rdf import audit_experience

for name, path in ttl_paths.items():
    questions = audit_experience(path)
    print(f"── {name}  ({path.name})  —  {len(questions)} question(s) ──")
    for q in questions:
        print(f"  [{q.slug}]  {q.field}")
        print(f"    → {q.question}")
    print()

── frodo  (frodo.ttl)  —  2 question(s) ──
  [wh_shire_2020]  endDate
    → When did you leave Shire Consulting Group, or is this your current role (YYYY-MM-DD or 'present')?
  [wh_pelennor_infra_2025]  endDate
    → When did you leave Pelennor Infrastructure Partners, or is this your current role (YYYY-MM-DD or 'present')?

── sam_v1  (sam_v1.ttl)  —  1 question(s) ──
  [wh_shire_2020]  endDate
    → When did you leave Shire Consulting Group, or is this your current role (YYYY-MM-DD or 'present')?

── sam_v2  (sam_v2.ttl)  —  1 question(s) ──
  [wh_rivendell_health_2025]  endDate
    → When did you leave Rivendell Health Systems, or is this your current role (YYYY-MM-DD or 'present')?



### 2.2 — Inspect Frodo's questions as a DataFrame

In [31]:
import pandas as pd

qs = audit_experience(ttl_paths["frodo"])
df = pd.DataFrame([
    {"slug": q.slug, "field": q.field, "question": q.question}
    for q in qs
])
df

,slug,field,question
0,wh_shire_2020,endDate,"When did you leave Shire Consulting Group, or ..."
1,wh_pelennor_infra_2025,endDate,When did you leave Pelennor Infrastructure Par...


### 2.3 — Fill in a missing value with `update_field`

`update_field(ttl_file, slug_or_iri, field, value)` resolves the node by
slug (the local part of its IRI), removes the old triple if present,
adds the new value with correct RDF datatype, and saves in-place.

We work on a copy so the cached original stays intact.

In [32]:
from resume_rdf import update_field

qa_dir = SHIRE_DIR /Path("shire_qa")
qa_dir.mkdir(exist_ok=True)
frodo_qa = qa_dir / "frodo.ttl"
shutil.copy(ttl_paths["frodo"], frodo_qa)

# Baseline question count
before = audit_experience(frodo_qa)
print(f"Before: {len(before)} question(s)")
for q in before:
    print(f"  [{q.slug}]  {q.field}")

Before: 2 question(s)
  [wh_shire_2020]  endDate
  [wh_pelennor_infra_2025]  endDate


In [33]:
# Answer the first open question
# In practice a user (or a conversational loop) would supply the real value;
# here we use plausible stand-ins keyed by field name.
sample_values = {
    "jobDescription":    "Led strategic consulting engagements for energy and financial services clients.",
    "endDate":           "2023-06-30",
    "startDate":         "2021-01-01",
    "jobTitle":          "Senior Consultant",
    "employedIn":        "http://example.org/cv/company_shire_consulting",
    "benefitsDelivered": "Delivered a 14% reduction in outage duration and secured 12,000 demand-response contracts.",
    "activitiesPerformed": "Managed programme governance, ran stakeholder workshops, and reviewed technical architecture.",
    "roleTitle":         "Programme Lead",
    "projectDescription":"Smart grid modernisation covering AMI rollout, SCADA integration, and demand-side response.",
    "usesSkill":         "(use update_field with the skill IRI — see note below)",
}

filled = []
for q in before:
    if q.field in sample_values and q.field != "usesSkill":
        value = sample_values[q.field]
        update_field(frodo_qa, q.slug, q.field, value)
        filled.append((q.slug, q.field, value))
        print(f"  set [{q.slug}].{q.field} = {value!r}")

print(f"\nFilled {len(filled)} field(s).")
print("Note: usesSkill requires a URIRef to an existing :skill_* node — handled separately.")

  set [wh_shire_2020].endDate = '2023-06-30'
  set [wh_pelennor_infra_2025].endDate = '2023-06-30'

Filled 2 field(s).
Note: usesSkill requires a URIRef to an existing :skill_* node — handled separately.


### 2.4 — Re-run the audit to confirm resolution

In [34]:
after = audit_experience(frodo_qa)
print(f"Before: {len(before)} question(s)")
print(f"After:  {len(after)} question(s)")

remaining  = {(q.slug, q.field) for q in after}
resolved   = [(q.slug, q.field) for q in before if (q.slug, q.field) not in remaining]
still_open = [(q.slug, q.field) for q in after]

print(f"\nResolved ({len(resolved)}):")
for slug, field in resolved:
    print(f"  ✓ [{slug}]  {field}")

if still_open:
    print(f"\nStill open ({len(still_open)}):")
    for slug, field in still_open:
        print(f"  ? [{slug}]  {field}")

Before: 2 question(s)
After:  0 question(s)

Resolved (2):
  ✓ [wh_shire_2020]  endDate
  ✓ [wh_pelennor_infra_2025]  endDate


### 2.5 — Compare Sam v1 vs v2 audit profiles

The two versions of Sam's CV will likely produce different question sets —
v2 (ESG framing) tends to be more verbose in project descriptions, so it
may have fewer `projectDescription` and `benefitsDelivered` gaps.
This cell shows the field-level diff.

In [35]:
q_v1 = {(q.slug, q.field) for q in audit_experience(ttl_paths["sam_v1"])}
q_v2 = {(q.slug, q.field) for q in audit_experience(ttl_paths["sam_v2"])}

only_in_v1 = q_v1 - q_v2
only_in_v2 = q_v2 - q_v1
in_both    = q_v1 & q_v2

print(f"Gaps in v1 only ({len(only_in_v1)} — filled by v2 phrasing):")
for slug, field in sorted(only_in_v1):
    print(f"  [{slug}]  {field}")

print(f"\nGaps in v2 only ({len(only_in_v2)} — filled by v1 phrasing):")
for slug, field in sorted(only_in_v2):
    print(f"  [{slug}]  {field}")

print(f"\nGaps in both versions ({len(in_both)}):")
for slug, field in sorted(in_both):
    print(f"  [{slug}]  {field}")

Gaps in v1 only (1 — filled by v2 phrasing):
  [wh_shire_2020]  endDate

Gaps in v2 only (1 — filled by v1 phrasing):
  [wh_rivendell_health_2025]  endDate

Gaps in both versions (0):


### 2.6 — Export questions as JSON (for downstream tooling)

The `Question` dataclass is a plain frozen dataclass — easy to serialise.
This is the shape expected by any conversational loop that picks up the
questions and presents them to the CV owner.

In [37]:
import json
from resume_rdf import Question

all_questions = {}
for name, path in ttl_paths.items():
    qs = audit_experience(path)
    all_questions[name] = [
        {"slug": q.slug, "field": q.field, "question": q.question}
        for q in qs
    ]

out_path = SHIRE_DIR / Path("shire_qa") / "questions.json"
out_path.parent.mkdir(exist_ok=True)
out_path.write_text(json.dumps(all_questions, indent=2))
print(f"Wrote {sum(len(v) for v in all_questions.values())} questions to {out_path}")
print()
print(json.dumps(all_questions, indent=2)[:800], "...")

Wrote 4 questions to shire/shire_qa/questions.json

{
  "frodo": [
    {
      "slug": "wh_shire_2020",
      "field": "endDate",
      "question": "When did you leave Shire Consulting Group, or is this your current role (YYYY-MM-DD or 'present')?"
    },
    {
      "slug": "wh_pelennor_infra_2025",
      "field": "endDate",
      "question": "When did you leave Pelennor Infrastructure Partners, or is this your current role (YYYY-MM-DD or 'present')?"
    }
  ],
  "sam_v1": [
    {
      "slug": "wh_shire_2020",
      "field": "endDate",
      "question": "When did you leave Shire Consulting Group, or is this your current role (YYYY-MM-DD or 'present')?"
    }
  ],
  "sam_v2": [
    {
      "slug": "wh_rivendell_health_2025",
      "field": "endDate",
      "question": "When did you leave Rivendell Health Systems, or is this your current r ...


---
## Summary

| Feature | Function | CLI |
|---------|----------|-----|
| CV → Turtle RDF | `generate_graph_from_file` / `generate_graph_from_bytes` | `cv-to-rdf` |
| Extract entities | `load_entities(ttl_files)` | — |
| Find near-duplicate pairs | `find_matches(entities, threshold)` | — |
| Interactive reconciliation | `reconcile_interactive(ttl_files)` | `cv-reconcile` |
| Apply IRI mapping | `apply_mapping(ttl_files, mapping)` | — |
| Audit for missing fields | `audit_experience(ttl_file)` | `cv-audit` |
| Update a field | `update_field(ttl_file, slug, field, value)` | `cv-update` |

The `shire/` test CVs are designed so that:
- **Cross-person** reconciliation: 3 shared projects appear in Frodo and Sam's files
- **Within-person** reconciliation: Sam's v1 and v2 describe the same projects differently
- **Audit coverage**: both CVs will produce at least some open questions (parsers
  occasionally omit optional fields), letting the update → re-audit loop be demonstrated
  with real data